In [ ]:
from pathlib import Path
import base64
import os
import shutil
import subprocess
import sys
import zipfile

# VS Code에서 Colab 커널에 연결해서 실행하는 전용 셀입니다.
# 업로드 위젯을 쓰지 않으므로 파일 선택 창이 뜨지 않습니다.
BUNDLE_B64 = (
    "..."
)

workdir = Path("/content/RAG_chatbot_colab")
zip_path = Path("/content/colab_train_bundle.zip")
shutil.rmtree(workdir, ignore_errors=True)
workdir.mkdir(parents=True, exist_ok=True)
zip_path.write_bytes(base64.b64decode(BUNDLE_B64))
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(workdir)
os.chdir(workdir)
print(f"학습 파일 준비 완료: {workdir}")

gpu_check = subprocess.run(
    ["bash", "-lc", "command -v nvidia-smi >/dev/null && nvidia-smi -L"],
    text=True,
    capture_output=True,
)
has_gpu = gpu_check.returncode == 0 and "GPU" in gpu_check.stdout
device = "cuda" if has_gpu else "cpu"
print(gpu_check.stdout.strip() or "GPU를 찾지 못했습니다. CPU로 실행합니다.")

batch_size = 64 if device == "cuda" else 16
epochs = 6
block_size = 256
embedding_dim = 384
num_heads = 8
num_layers = 8
learning_rate = 1.5e-4
min_learning_rate = 1.5e-5
warmup_steps = 1000
eval_every = 500
eval_batches = 50

print(
    f"학습 시작: device={device}, batch_size={batch_size}, epochs={epochs}, "
    f"block_size={block_size}, dim={embedding_dim}, layers={num_layers}"
)
print("PyTorch/CUDA 첫 로딩은 1-3분 걸릴 수 있습니다. 여기서 중지하지 마세요.")

cmd = [
    sys.executable, "-m", "chatbot.train",
    "--corpus", "chatbot/autocomplete_corpus_nikl.txt",
    "--output", "artifacts/chatbot.pt",
    "--device", device,
    "--epochs", str(epochs),
    "--max-steps", "0",
    "--batch-size", str(batch_size),
    "--learning-rate", str(learning_rate),
    "--min-learning-rate", str(min_learning_rate),
    "--warmup-steps", str(warmup_steps),
    "--eval-every", str(eval_every),
    "--eval-batches", str(eval_batches),
    "--block-size", str(block_size),
    "--embedding-dim", str(embedding_dim),
    "--num-heads", str(num_heads),
    "--num-layers", str(num_layers),
]
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run(cmd, check=True, env=env)

model_path = workdir / "artifacts" / "chatbot.pt"
easy_path = Path("/content/chatbot.pt")
shutil.copy2(model_path, easy_path)
print(f"학습 완료: {model_path}")
print(f"VS Code Colab 파일 브라우저에서 받기 쉬운 복사본: {easy_path}")

drive_dir = Path("/content/drive/MyDrive")
if drive_dir.exists():
    drive_copy = drive_dir / "chatbot.pt"
    shutil.copy2(easy_path, drive_copy)
    print(f"Google Drive에도 복사했습니다: {drive_copy}")
else:
    print("Google Drive가 마운트되어 있지 않아서 /content/chatbot.pt에만 저장했습니다.")
